# Notebook 02 — Feature Engineering Pipeline
## Pearls AQI Predictor · Hyderabad, Pakistan

**Objective:** Load the merged hourly dataset and build the complete feature set used for model training. This notebook demonstrates every feature group created by `FeatureBuilder`, explains the rationale behind each group, and visualizes the resulting feature distributions.

**Feature groups covered:**
1. **Time features** — hour, day, month, weekend, season, cyclical encodings
2. **Lag features** — AQI/PM at t-1, t-6, t-24, t-72
3. **Rolling statistics** — mean, std, min, max over 6h and 24h windows
4. **Weather features** — temperature, humidity, wind, pressure, precipitation, cloud cover
5. **Interaction features** — humidity × temperature, wind × PM2.5, rain × PM10, AQI change rate
6. **Targets** — AQI at t+24h, t+48h, t+72h
7. **Classification labels** — Good, Moderate, Unhealthy, etc.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from feature_store.feature_builder import FeatureBuilder, build_features
from utils.config import get
from utils.storage import load_parquet, save_parquet
from utils.time_utils import now_local

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 300)

In [ ]:
# Load merged hourly data
DATA_DIR = Path(get('storage.data_dir', '../data'))
merged_path = DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet'

try:
    df = load_parquet(merged_path)
    print(f'✅ Loaded {len(df)} rows × {len(df.columns)} columns')
    print(f'   Timestamp range: {df["timestamp"].min()} → {df["timestamp"].max()}')
except FileNotFoundError:
    print('❌ No merged data found. Run Notebook 01 first.')
    # Create synthetic data for demonstration
    from datetime import datetime, timedelta
    base = datetime.now().replace(minute=0, second=0, microsecond=0)
    np.random.seed(42)
    n = 300
    df = pd.DataFrame({
        'timestamp': [base - timedelta(hours=i) for i in range(n, 0, -1)],
        'aqi': np.clip(60 + np.random.randn(n) * 25 + np.sin(np.arange(n) * 2 * np.pi / 24) * 15, 0, 300),
        'pm2_5': np.clip(40 + np.random.randn(n) * 15, 0, 200),
        'pm10': np.clip(60 + np.random.randn(n) * 25, 0, 300),
        'temperature_2m': 25 + np.sin(np.arange(n) * 2 * np.pi / 24) * 8 + np.random.randn(n) * 2,
        'relative_humidity_2m': np.clip(50 + np.random.randn(n) * 15, 0, 100),
        'pressure_msl': 1010 + np.random.randn(n) * 5,
        'wind_speed_10m': np.abs(3 + np.random.randn(n) * 2),
        'wind_direction_10m': np.random.uniform(0, 360, n),
        'precipitation': np.abs(np.random.randn(n)) * 3,
        'cloud_cover': np.clip(np.random.uniform(0, 100, n), 0, 100),
        'dew_point_2m': 15 + np.random.randn(n) * 5,
    })
    print(f'⚠️  Using {n} rows of synthetic data for demonstration')

---
## 1. Run Full Feature Pipeline

The `FeatureBuilder` class applies all feature groups in sequence. Let's inspect each group.

In [ ]:
builder = FeatureBuilder(df)
featured = builder.build_all()

# Categorize all generated columns
all_cols = list(featured.columns)

time_cols = [c for c in all_cols if c in ('hour','day','day_of_week','month','weekend','season','hour_sin','hour_cos','month_sin','month_cos','day_of_week_sin','day_of_week_cos')]
lag_cols = [c for c in all_cols if 'lag_' in c]
rolling_cols = [c for c in all_cols if 'roll_' in c]
weather_cols = [c for c in all_cols if c in ('temperature_2m','relative_humidity_2m','dew_point_2m','pressure_msl','wind_speed_10m','wind_direction_10m','precipitation','cloud_cover')]
interaction_cols = [c for c in all_cols if 'humidity_x' in c or 'wind_x' in c or 'rain_x' in c or 'change_rate' in c]
target_cols = [c for c in all_cols if c.startswith('target_') and not c.endswith('_category')]
category_cols = [c for c in all_cols if c.endswith('_category')]

print(f'=== Feature Groups ===')
print(f'Total columns: {len(all_cols)}')
print(f'  Time features:      {len(time_cols):>2}  → {time_cols}')
print(f'  Lag features:       {len(lag_cols):>2}  → {lag_cols}')
print(f'  Rolling statistics: {len(rolling_cols):>2}  → {rolling_cols}')
print(f'  Weather features:   {len(weather_cols):>2}  → {weather_cols}')
print(f'  Interaction feats:  {len(interaction_cols):>2}  → {interaction_cols}')
print(f'  Target columns:     {len(target_cols):>2}  → {target_cols}')
print(f'  Category labels:    {len(category_cols):>2}  → {category_cols}')
print(f'  Other (metadata):   {len(all_cols) - sum(len(g) for g in [time_cols,lag_cols,rolling_cols,weather_cols,interaction_cols,target_cols,category_cols])}')

---
## 2. Time Features — Visualizing Cyclical Encodings

Instead of encoding hour as 0–23 (where 23 and 0 are far apart in linear space but adjacent in reality),
we use `sin` and `cos` transforms to preserve the circular nature of time.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hour circular encoding
hours = np.arange(0, 24)
hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

axes[0].scatter(hour_cos, hour_sin, c=hours, cmap='twilight', s=100, edgecolors='white')
for i, h in enumerate(hours):
    axes[0].annotate(str(h), (hour_cos[i]+0.03, hour_sin[i]+0.03), fontsize=9, color='white')
axes[0].set_title('Hour: sin/cos encoding (24h cycle)')
axes[0].set_xlabel('cos(hour)')
axes[0].set_ylabel('sin(hour)')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.3)
axes[0].set_aspect('equal')

# Month circular encoding
months = np.arange(1, 13)
axes[1].scatter(np.cos(2*np.pi*months/12), np.sin(2*np.pi*months/12), c=months, cmap='coolwarm', s=100, edgecolors='white')
for i, m in enumerate(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']):
    axes[1].annotate(m, (np.cos(2*np.pi*months[i]/12)+0.03, np.sin(2*np.pi*months[i]/12)+0.03), fontsize=9, color='white')
axes[1].set_title('Month: sin/cos encoding (12-month cycle)')
axes[1].set_aspect('equal')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.3)

# AQI by hour of day (from actual data)
if 'aqi' in featured.columns:
    hourly_avg = featured.groupby('hour')['aqi'].mean()
elif 'om_forecast_aqi' in featured.columns:
    hourly_avg = featured.groupby('hour')['om_forecast_aqi'].mean()
else:
    hourly_avg = pd.Series(index=range(24), data=np.zeros(24))

axes[2].bar(range(24), hourly_avg.values, color=plt.cm.viridis(np.linspace(0.2, 0.8, 24)))
axes[2].set_title('Average AQI by Hour of Day')
axes[2].set_xlabel('Hour')
axes[2].set_ylabel('Average AQI')
axes[2].set_xticks(range(0, 24, 3))

plt.tight_layout()
plt.show()

---
## 3. Lag Features — Autocorrelation Structure

Lag features tell the model "what was AQI 1 hour ago? 6 hours? yesterday?". These capture temporal dependencies.

In [ ]:
aqi_col = 'aqi' if 'aqi' in featured.columns else 'om_forecast_aqi'

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, lag in zip(axes, [1, 6, 24, 72]):
    lag_col = f'aqi_lag_{lag}'
    if lag_col in featured.columns and aqi_col in featured.columns:
        valid = featured[[aqi_col, lag_col]].dropna()
        ax.scatter(valid[lag_col], valid[aqi_col], alpha=0.4, s=10, color='#00d4ff')
        corr = valid[lag_col].corr(valid[aqi_col])
        ax.set_title(f'Lag {lag}h (r={corr:.3f})')
        ax.set_xlabel(f'AQI at t-{lag}h')
        ax.set_ylabel('AQI at t')
        # Add y=x line
        lims = [min(valid.min().min(), 0), max(valid.max().max(), 100)]
        ax.plot(lims, lims, 'r--', alpha=0.3, linewidth=1)
        ax.grid(True, alpha=0.2)

plt.suptitle('Lag Feature Correlation with Current AQI', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Full autocorrelation plot
if aqi_col in featured.columns:
    series = featured[aqi_col].dropna()
    max_lag = 96
    autocorrs = [series.autocorr(lag=lag) for lag in range(1, max_lag + 1)]

    fig, ax = plt.subplots(figsize=(16, 5))
    colors = ['#00e400' if v > 0 else '#ff3333' for v in autocorrs]
    ax.bar(range(1, max_lag + 1), autocorrs, color=colors, alpha=0.7, width=0.8)
    ax.axhline(y=0, color='white', linewidth=0.5)
    ax.axvline(x=24, color='#ff7e00', linestyle='--', linewidth=1.5, label='24h lag')
    ax.axvline(x=48, color='#ff0000', linestyle='--', linewidth=1.5, label='48h lag')
    ax.axvline(x=72, color='#8f3f97', linestyle='--', linewidth=1.5, label='72h lag')
    ax.set_title('AQI Autocorrelation by Lag (Hours)', fontsize=14)
    ax.set_xlabel('Lag (hours)')
    ax.set_ylabel('Pearson Correlation')
    ax.legend()
    ax.grid(True, alpha=0.15)
    plt.tight_layout()
    plt.show()

---
## 4. Rolling Statistics — Trend & Volatility

Rolling features capture short-term trends (6h) and daily patterns (24h).

In [ ]:
roll_cols = [c for c in featured.columns if 'roll_' in c]
if roll_cols and aqi_col in featured.columns:
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))

    # Raw AQI + rolling mean
    sample = featured.tail(150).copy()
    axes[0,0].plot(sample['timestamp'], sample[aqi_col], label='Raw AQI', alpha=0.5, linewidth=0.8)
    if 'aqi_roll_mean_6' in sample.columns:
        axes[0,0].plot(sample['timestamp'], sample['aqi_roll_mean_6'], label='6h Rolling Mean', linewidth=2)
    if 'aqi_roll_mean_24' in sample.columns:
        axes[0,0].plot(sample['timestamp'], sample['aqi_roll_mean_24'], label='24h Rolling Mean', linewidth=2)
    axes[0,0].set_title('AQI with Rolling Means')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.2)

    # Rolling std (volatility)
    if 'aqi_roll_std_24' in sample.columns:
        axes[0,1].fill_between(sample.index, sample['aqi_roll_std_24'], alpha=0.3, color='#ff7e00')
        axes[0,1].plot(sample.index, sample['aqi_roll_std_24'], color='#ff7e00', linewidth=1.5)
        axes[0,1].set_title('AQI Volatility (24h Rolling Std)')
        axes[0,1].set_ylabel('Standard Deviation')

    # Rolling min/max band
    if 'aqi_roll_min_24' in sample.columns and 'aqi_roll_max_24' in sample.columns:
        axes[1,0].fill_between(sample.index, sample['aqi_roll_min_24'], sample['aqi_roll_max_24'], 
                               alpha=0.25, color='#4da6ff', label='24h Min-Max Range')
        axes[1,0].plot(sample.index, sample[aqi_col], alpha=0.6, linewidth=0.8, label='Raw AQI')
        axes[1,0].set_title('24h AQI Range (Min-Max Band)')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

---
## 5. Interaction Features — Derived Signals

These combine multiple raw features to capture known physical relationships.

In [ ]:
interaction_cols = [c for c in featured.columns if any(x in c for x in ['humidity_x', 'wind_x', 'rain_x', 'change_rate'])]

if interaction_cols:
    fig, axes = plt.subplots(1, len(interaction_cols), figsize=(6 * len(interaction_cols), 4))
    if len(interaction_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, interaction_cols):
        sample = featured[col].dropna().tail(200)
        ax.plot(sample.index, sample.values, linewidth=1.5, color='#00d4ff')
        ax.set_title(col.replace('_', ' ').title())
        ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    # Show formulas
    print('Interaction feature formulas:')
    print('  humidity_x_temperature = relative_humidity_2m × temperature_2m')
    print('  wind_x_pm25            = wind_speed_10m × PM2.5')
    print('  rain_x_pm10            = precipitation × PM10')
    print('  aqi_change_rate        = ΔAQI over last 6 hours ÷ 6')

---
## 6. Target Construction — Future AQI Labels

The targets are shifted AQI values: at time `t`, `target_aqi_24h` = AQI at `t+24`.
This creates the supervised learning setup: given features at time `t`, predict AQI at `t+N`.

In [ ]:
target_cols = [c for c in featured.columns if c.startswith('target_') and not c.endswith('_category')]

if target_cols and aqi_col in featured.columns:
    # Verify shift correctness: target_aqi_24h at t=0 should match aqi at t=24
    print('=== Target Shift Verification ===')
    for target in target_cols:
        h = int(target.split('_')[-1].replace('h',''))
        t0_target = featured[target].iloc[0]
        tN_aqi = featured[aqi_col].iloc[h] if h < len(featured) else None
        if pd.notna(t0_target) and pd.notna(tN_aqi):
            match = abs(t0_target - tN_aqi) < 0.01
            print(f'  {target}: value at t=0 = {t0_target:.1f}, AQI at t={h} = {tN_aqi:.1f} → {"✅ CORRECT" if match else "❌ MISMATCH"}')
        else:
            print(f'  {target}: insufficient data for verification')

    # Distribution of targets
    fig, axes = plt.subplots(1, len(target_cols), figsize=(6 * len(target_cols), 4))
    if len(target_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, target_cols):
        vals = featured[col].dropna()
        ax.hist(vals, bins=30, color='#00d4ff', alpha=0.7, edgecolor='white', linewidth=0.5)
        ax.axvline(vals.mean(), color='#ff7e00', linestyle='--', linewidth=2, label=f'Mean: {vals.mean():.1f}')
        ax.set_title(f'{col}\n({len(vals)} non-null)')
        ax.set_xlabel('AQI')
        ax.legend()

    plt.suptitle('Target Distributions (Future AQI)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 7. Classification Labels — AQI Categories

Each target gets a category label. This is a lightweight classification layer on top of regression.

In [ ]:
from utils.aqi_utils import classify_aqi, AQICategory

cat_cols = [c for c in featured.columns if c.endswith('_category')]

if cat_cols:
    for col in cat_cols:
        counts = featured[col].value_counts()
        print(f'\n{col}:')
        for cat, cnt in counts.items():
            pct = cnt / len(featured) * 100
            print(f'  {cat}: {cnt} ({pct:.1f}%)')

    # Visualize
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 4))
    if len(cat_cols) == 1:
        axes = [axes]

    category_order = [e.value for e in AQICategory if e != AQICategory.UNKNOWN]
    category_colors = ['#00e400', '#ffff00', '#ff7e00', '#ff0000', '#8f3f97', '#7e0023']

    for ax, col in zip(axes, cat_cols):
        counts = featured[col].value_counts()
        ordered = {c: counts.get(c, 0) for c in category_order if c in counts.index}
        ax.bar(ordered.keys(), ordered.values(), color=category_colors[:len(ordered)])
        ax.set_title(col.replace('target_', '').replace('_category', ''))
        ax.tick_params(axis='x', rotation=45)

    plt.suptitle('AQI Category Distribution per Target Horizon', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 8. Feature Correlation Matrix

Understand which features correlate with each other and with the targets.

In [ ]:
# Select numeric features + targets for correlation
target_and_features = target_cols[:3] + [c for c in all_cols if c not in target_cols + cat_cols + ['timestamp','source','station_name','city','country','dominant_pollutant','merged_at','fetched_at','latitude','longitude']]
numeric_cols = [c for c in target_and_features if c in featured.columns and featured[c].dtype in (np.float64, np.float32, np.int64, np.int32)]

# Limit to top 25 by correlation with first target
if numeric_cols and target_cols:
    corr_with_target = featured[numeric_cols].corr()[target_cols[0]].abs().sort_values(ascending=False).head(25)
    top_cols = list(corr_with_target.index)

    fig, ax = plt.subplots(figsize=(14, 12))
    corr_matrix = featured[top_cols].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                ax=ax, square=True, linewidths=0.5, cbar_kws={'shrink': 0.7})
    ax.set_title('Feature Correlation Matrix (Top 25 Features by |correlation with first target|)', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 9. Save Feature Table

Persist the feature-engineered dataset for model training.

In [ ]:
from datetime import datetime

feature_path = DATA_DIR / 'processed' / 'features' / f'features_nb_{datetime.now().strftime("%Y%m%d_%H%M")}.parquet'
save_parquet(featured, feature_path)
print(f'✅ Feature table saved: {feature_path}')
print(f'   {len(featured)} rows × {len(featured.columns)} columns')

# Also save training-ready dataset (drops rows without targets)
train_df = builder.get_training_data()
train_path = DATA_DIR / 'processed' / 'features' / f'train_nb_{datetime.now().strftime("%Y%m%d_%H%M")}.parquet'
save_parquet(train_df, train_path)
print(f'✅ Training dataset saved: {train_path}')
print(f'   {len(train_df)} rows (non-NaN targets only)')

---
## Summary

| Step | Result |
|------|--------|
| Time features | 12 columns: hour, day, month, weekend, season, cyclical sin/cos pairs |
| Lag features | 12 columns: AQI/PM2.5/PM10 at t-1, t-6, t-24, t-72 |
| Rolling stats | 8 columns: mean/std/min/max over 6h and 24h |
| Weather | 8 columns: temp, humidity, pressure, wind, precipitation, cloud |
| Interactions | 4 columns: humidity×temp, wind×PM2.5, rain×PM10, ΔAQI |
| Targets | 3-5 columns: target_aqi_24h/48h/72h + optional PM targets |
| Classification | Category labels for each target horizon |

**Next:** Notebook 03 — Model Training & Comparison